In [ ]:
import pandas as pd
import numpy as np
import os

# ============================
# CONFIGURATION
# ============================
# Input file name (ensure exact match with the file name in your directory)
input_file = "TNBC_patient_GSM1589063_vs_normals.csv"
output_file = "logFC_patient_41.csv"

# ============================
# EXECUTION
# ============================

def calculate_logfc():
    print(f"--- Initiating LogFC calculation for {input_file} ---")
    
    # 1. Data Loading
    if not os.path.exists(input_file):
        print(f"❌ Error: The file {input_file} does not exist in the directory.")
        return

    df = pd.read_csv(input_file)
    
    # 2. Column Identification
    # Expected structure: [GeneSymbol, PatientID, Normal1, Normal2, ...]
    gene_col_name = df.columns[0]      # Column 0: Gene Symbols
    patient_col_name = df.columns[1]   # Column 1: TNBC Patient
    normal_cols = df.columns[2:]       # Column 2 onwards: Normal Samples
    
    print(f"   > Identified patient: {patient_col_name}")
    print(f"   > Number of normal samples utilized for comparison: {len(normal_cols)}")
    
    # 3. Preliminary Data Assessment (Log vs. Linear Scale)
    # If the maximum values are < 50, the data are likely already log-transformed (GEO standard).
    # If the values are large (e.g., 2000, 50000), they represent raw counts and log transformation is required.
    max_val = df[patient_col_name].max()
    if max_val > 50:
        print(f"⚠️  WARNING: Data values appear to be raw counts (max={max_val}).")
        print("    Applying log2 transformation prior to subtraction.")
        # Offset added (+1) to prevent log(0) calculation
        patient_vals = np.log2(df[patient_col_name] + 1)
        normal_vals = np.log2(df[normal_cols] + 1)
    else:
        print(f"   > Data appear to be log-normalized (max={max_val}). Proceeding with direct subtraction.")
        patient_vals = df[patient_col_name]
        normal_vals = df[normal_cols]

    # 4. Computation
    # Compute the mean value of normal samples for each row (gene)
    mean_normals = normal_vals.mean(axis=1)
    
    # Compute LogFC: (Patient Value) - (Mean of Normal Samples)
    logfc_values = patient_vals - mean_normals
    
    # 5. Output DataFrame Construction
    result_df = pd.DataFrame({
        "Patient_ID": patient_col_name, # Identical ID across all records
        "GeneSymbol": df[gene_col_name],
        "logFC": logfc_values
    })
    
    # 6. Descending Sort
    result_df = result_df.sort_values(by="logFC", ascending=False)
    
    # 7. File Serialization
    result_df.to_csv(output_file, index=False)
    
    print(f"✅ Computation finalized.")
    print(f"    File successfully saved: {output_file}")
    print(f"    Top 5 rows:\n")
    print(result_df.head().to_string(index=False))

# Function Invocation
if __name__ == "__main__":
    calculate_logfc()